In [22]:
from neupan import neupan
import irsim
import numpy as np
import argparse

import warnings
# implement the code update in the imported files immediately
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [23]:
example_folder = "/home/pengchen/Research/NeuPAN/example"
env_file = f"{example_folder}/mosaic_nav/acker/env.yaml"
planner_file = f"{example_folder}/mosaic_nav/acker/planner.yaml"
env = irsim.make(env_file)
neupan_planner = neupan.init_from_yaml(planner_file)


2025-10-26 14:28 | WARNING  | None not found
2025-10-26 14:28 | INFO     | Simulation environment 'env' has been initialized and started.


In [24]:
robot_state = env.get_robot_state()
robot_vel = env.get_robot_velocity()
lidar_scan = env.get_lidar_scan()
points = neupan_planner.scan_to_point(robot_state, lidar_scan)
print(f"robot_state: {robot_state}")

robot_state: [[10.  ]
 [42.  ]
 [ 1.57]
 [ 0.  ]]


In [25]:
print(f"Neupan planner robot mosaic ratios: {neupan_planner.robot.mosaic_ratios}")
print(f"Neupan planner robot translations: {neupan_planner.robot.mosaic_translations}")

Neupan planner robot mosaic ratios: tensor([1.0125, 1.0125, 1.0125, 0.6750, 0.5625, 0.5625, 0.4500, 0.4500, 0.4500,
        0.2250])
Neupan planner robot translations: tensor([[-1.4625, -0.1125],
        [ 0.5625, -0.1125],
        [ 1.2375, -0.1125],
        [-2.0250, -0.2250],
        [-2.3625, -0.1125],
        [-1.6875, -0.7875],
        [-2.7000, -0.2250],
        [-1.5750, -1.3500],
        [-1.5750,  0.9000],
        [-1.3500, -1.8000]])


In [26]:
import torch
import numpy as np

device = torch.device("cpu")
time_print = False
log_cost = False

def np_to_tensor(array):
        
    if np.isscalar(array):
        return torch.tensor(array).type(torch.float32).to(device)

    return torch.from_numpy(array).type(torch.float32).to(device)

In [ ]:
neupan_planner.ipath.check_arrive(robot_state)
nom_input_np = neupan_planner.ipath.generate_nom_ref_state(
            robot_state, neupan_planner.cur_vel_array, neupan_planner.ref_speed
        )
nom_input_tensor = [np_to_tensor(n) for n in nom_input_np]
nom_input_tensor
# nom_s, nom_u, ref_s, ref_us

[tensor([[10.0000, 10.0000, 10.0000, 10.0000, 10.0000, 10.0000, 10.0000, 10.0000,
          10.0000, 10.0000, 10.0000],
         [42.0000, 42.0000, 42.0000, 42.0000, 42.0000, 42.0000, 42.0000, 42.0000,
          42.0000, 42.0000, 42.0000],
         [ 1.5700,  1.5700,  1.5700,  1.5700,  1.5700,  1.5700,  1.5700,  1.5700,
           1.5700,  1.5700,  1.5700]]),
 tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]]),
 tensor([[10.0000, 10.0269, 10.1067, 10.2377, 10.4179, 10.6438, 10.9116, 11.2164,
          11.5530, 11.9152, 12.2966],
         [42.0000, 42.3988, 42.7905, 43.1681, 43.5249, 43.8546, 44.1513, 44.4099,
          44.6255, 44.7946, 44.9140],
         [ 1.5700,  1.4367,  1.3033,  1.1700,  1.0367,  0.9033,  0.7700,  0.6367,
           0.5033,  0.3700,  0.2367]]),
 tensor([4., 4., 4., 4., 4., 4., 4., 4., 4., 4.])]

In [28]:
from pan import _norm_ratios, _norm_translations
normed_ratios = _norm_ratios(neupan_planner.robot.mosaic_ratios,
                             dtype=nom_input_tensor[0].dtype,
                             device=nom_input_tensor[0].device)
print(f"Normalized Ratios: {normed_ratios}")
print(f"Original Ratios: {neupan_planner.robot.mosaic_ratios}")

normed_translations = _norm_translations(neupan_planner.robot.mosaic_translations,
                             dtype=nom_input_tensor[0].dtype,
                             device=nom_input_tensor[0].device)
print(f"Normalized Ratios: {normed_translations}")
print(f"Original Ratios: {neupan_planner.robot.mosaic_translations}")

Normalized Ratios: tensor([1.0125, 1.0125, 1.0125, 0.6750, 0.5625, 0.5625, 0.4500, 0.4500, 0.4500,
        0.2250])
Original Ratios: tensor([1.0125, 1.0125, 1.0125, 0.6750, 0.5625, 0.5625, 0.4500, 0.4500, 0.4500,
        0.2250])
Normalized Ratios: tensor([[-1.4625, -0.1125],
        [ 0.5625, -0.1125],
        [ 1.2375, -0.1125],
        [-2.0250, -0.2250],
        [-2.3625, -0.1125],
        [-1.6875, -0.7875],
        [-2.7000, -0.2250],
        [-1.5750, -1.3500],
        [-1.5750,  0.9000],
        [-1.3500, -1.8000]])
Original Ratios: tensor([[-1.4625, -0.1125],
        [ 0.5625, -0.1125],
        [ 1.2375, -0.1125],
        [-2.0250, -0.2250],
        [-2.3625, -0.1125],
        [-1.6875, -0.7875],
        [-2.7000, -0.2250],
        [-1.5750, -1.3500],
        [-1.5750,  0.9000],
        [-1.3500, -1.8000]])


In [29]:
neupan_planner.robot.mosaic_ratios.numel()

10

In [ ]:
point_velocities =
action, info = neupan_planner(robot_state, points, point_velocities, robot_vel)


- nrmp forward execute time 0.008955 seconds
costs: {'total': 668.9776611328125, 's': 6.888127326965332, 'u': 424.14849853515625, 'prox': 6.787104211980477e-11, 'C1': -12.059001922607422, 'I': 250.0}
--------------------------------------------------
neupan forward execute time 0.009672 seconds
